In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing & Modeling ---
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
#   from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from xgboost import XGBClassifier

# --- Evaluation ---
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [2]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = "/content/drive/MyDrive/"
    print("Running in Google Colab")

except ModuleNotFoundError:
    base_path = "./"
    print("Running in Local Jupyter")

files = [
    "Borrower.txt",
    "Borrower_Prod.txt",
    "Loan.txt",
    "Loan_Prod.txt"
]

for file in files:
    path = os.path.join(base_path, file)
    if os.path.exists(path):
        print(f"{file} found")
    else:
        print(f"{file} not found")

Running in Local Jupyter
Borrower.txt found
Borrower_Prod.txt found
Loan.txt found
Loan_Prod.txt found


In [3]:
import pandas as pd
import os

file_name = "Loan_default.csv"

# Case 1: File is in the same folder (works in VS Code & Jupyter)
if os.path.exists(file_name):
    df = pd.read_csv(file_name)

# Case 2: Running in Google Colab with Google Drive
elif os.path.exists("/content/drive/MyDrive/" + file_name):
    df = pd.read_csv("/content/drive/MyDrive/" + file_name)

# Case 3: Ask user to upload file (Colab fallback)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(file_name)
    except:
        raise FileNotFoundError(
            f"{file_name} not found. Please place it in the project folder."
        )

print("Dataset Loaded Successfully")
print(df.head())

Dataset Loaded Successfully
      ID  year loan_limit             Gender approv_in_adv loan_type  \
0  24890  2019         cf  Sex Not Available         nopre     type1   
1  24891  2019         cf               Male         nopre     type2   
2  24892  2019         cf               Male           pre     type1   
3  24893  2019         cf               Male         nopre     type1   
4  24894  2019         cf              Joint           pre     type1   

  loan_purpose Credit_Worthiness open_credit business_or_commercial  ...  \
0           p1                l1        nopc                  nob/c  ...   
1           p1                l1        nopc                    b/c  ...   
2           p1                l1        nopc                  nob/c  ...   
3           p4                l1        nopc                  nob/c  ...   
4           p1                l1        nopc                  nob/c  ...   

   credit_type  Credit_Score  co-applicant_credit_type    age  \
0          EXP   

In [4]:
print("Dataset shape:", df.shape)
print("\nDataset info:")
df.info()
print("\nFirst 5 rows:")
df.head()

Dataset shape: (148670, 34)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148670 entries, 0 to 148669
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ID                         148670 non-null  int64  
 1   year                       148670 non-null  int64  
 2   loan_limit                 145326 non-null  object 
 3   Gender                     148670 non-null  object 
 4   approv_in_adv              147762 non-null  object 
 5   loan_type                  148670 non-null  object 
 6   loan_purpose               148536 non-null  object 
 7   Credit_Worthiness          148670 non-null  object 
 8   open_credit                148670 non-null  object 
 9   business_or_commercial     148670 non-null  object 
 10  loan_amount                148670 non-null  int64  
 11  rate_of_interest           112231 non-null  float64
 12  Interest_rate_spread       112031 non-null 

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [5]:
# List of columns to drop
cols_to_drop = [
    "ID",
    "year",
    "loan_limit",
    "open_credit",
    "business_or_commercial",
    "co-applicant_credit_type",
    "construction_type",
    "total_units",
    "Region",
    "Security_Type"
]

# Drop columns safely (ignore if some are missing)
df = df.drop(columns=cols_to_drop, errors='ignore')

print("Remaining columns:", df.columns.tolist())


Remaining columns: ['Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'property_value', 'occupancy_type', 'Secured_by', 'income', 'credit_type', 'Credit_Score', 'age', 'submission_of_application', 'LTV', 'Status', 'dtir1']


In [6]:
print(f"Original unique values in 'Status' column: {df['Status'].unique()}")

Original unique values in 'Status' column: [1 0]


In [7]:
df['Status'] = pd.to_numeric(df['Status'], errors='coerce')

In [8]:
valid_statuses = [0, 1]
df = df[df['Status'].isin(valid_statuses)]
print(f"Shape after keeping only rows with Status of 0 or 1: {df.shape}")

Shape after keeping only rows with Status of 0 or 1: (148670, 24)


In [9]:
df['Status'] = df['Status'].astype(int)

In [10]:
#daatset splitting
X = df.drop('Status', axis=1)
y = df['Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\nData successfully split.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("-" * 50)


Data successfully split.
X_train shape: (111502, 23)
X_test shape: (37168, 23)
--------------------------------------------------


In [11]:
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

print("\nPreprocessing pipeline built successfully.")
print("-" * 50)


Preprocessing pipeline built successfully.
--------------------------------------------------


In [15]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Handle class imbalance
scale_pos_weight_value = y_train.value_counts()[0] / y_train.value_counts()[1]

# Define models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),

    "Random Forest": RandomForestClassifier(random_state=42),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight_value
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=10),

    "Gaussian Naive Bayes": GaussianNB(),

    "Bernoulli Naive Bayes": BernoulliNB(),

    "Decision Tree": DecisionTreeClassifier(random_state=42)
}


# Train and evaluate
for name, model in models.items():

    print(f"\n--- Training and Evaluating: {name} ---")

    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])

    # Train
    model_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = model_pipeline.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f} (Note: Accuracy alone can be misleading for imbalanced data!)")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("-" * 50)


--- Training and Evaluating: Logistic Regression ---
Accuracy: 0.8409 (Note: Accuracy alone can be misleading for imbalanced data!)

Confusion Matrix:
[[24853  3155]
 [ 2757  6403]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.89     28008
           1       0.67      0.70      0.68      9160

    accuracy                           0.84     37168
   macro avg       0.79      0.79      0.79     37168
weighted avg       0.84      0.84      0.84     37168

--------------------------------------------------

--- Training and Evaluating: Random Forest ---
Accuracy: 1.0000 (Note: Accuracy alone can be misleading for imbalanced data!)

Confusion Matrix:
[[28008     0]
 [    0  9160]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28008
           1       1.00      1.00      1.00      9160

    accuracy                           1.00     37

In [16]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

results_list = []

print("\n--- Starting Final Evaluation on the Unseen Test Data ---")

for name, model in models.items():

    print(f"\n--- Training and Evaluating: {name} ---")

    # Create pipeline
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # Train
    model_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = model_pipeline.predict(X_test)

    # Probabilities (for ROC-AUC)
    if hasattr(model_pipeline, "predict_proba"):
        y_prob = model_pipeline.predict_proba(X_test)[:,1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = None

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"\nAccuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if roc_auc:
        print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("-"*60)

    # Store results
    results_list.append({
        "Model": name,
        "Accuracy": round(accuracy,4),
        "Precision": round(precision,4),
        "Recall": round(recall,4),
        "F1-score": round(f1,4),
        "ROC-AUC": round(roc_auc,4) if roc_auc else None
    })

print("\n--- Final Evaluation Complete ---")

# Results dataframe
results_df = pd.DataFrame(results_list)

# Sort by F1 score
results_df = results_df.sort_values(by="F1-score", ascending=False)

print("\nModel Comparison Table:")
print(results_df)


--- Starting Final Evaluation on the Unseen Test Data ---

--- Training and Evaluating: Logistic Regression ---

Accuracy : 0.8719
Precision: 0.9386
Recall   : 0.5139
F1-score : 0.6641
ROC-AUC  : 0.8590

Confusion Matrix:
[[27700   308]
 [ 4453  4707]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.99      0.92     28008
           1       0.94      0.51      0.66      9160

    accuracy                           0.87     37168
   macro avg       0.90      0.75      0.79     37168
weighted avg       0.88      0.87      0.86     37168

------------------------------------------------------------

--- Training and Evaluating: Random Forest ---

Accuracy : 1.0000
Precision: 0.9999
Recall   : 1.0000
F1-score : 0.9999
ROC-AUC  : 1.0000

Confusion Matrix:
[[28007     1]
 [    0  9160]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28008
           1

In [17]:
# --- ONE-TIME SCRIPT TO TRAIN AND SAVE PIPELINES ---

import joblib
import os
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

print("\n--- Starting Training and Serialization ---\n")

# Model save directory
save_path = os.path.join("..", "streamlit_app", "models")

# Create directory if it doesn't exist
os.makedirs(save_path, exist_ok=True)

for name, model in models.items():

    print(f"Training pipeline for: {name}")

    # Build pipeline
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])

    # Train model
    model_pipeline.fit(X_train, y_train)

    # Clean filename
    filename = name.lower().replace(" ", "_").replace("-", "_") + "_pipeline.joblib"

    # Full path
    full_path = os.path.join(save_path, filename)

    # Save model
    joblib.dump(model_pipeline, full_path)

    print(f"✔ Saved pipeline to: {os.path.abspath(full_path)}\n")

print("✅ All model pipelines trained and saved successfully!")

# Show directory
print("\nModels saved in:", os.path.abspath(save_path))

# Verify saved models
saved_files = sorted([f for f in os.listdir(save_path) if f.endswith(".joblib")])

print(f"\nSaved model files ({len(saved_files)}):")
for file in saved_files:
    print(" -", file)


--- Starting Training and Serialization ---

Training pipeline for: Logistic Regression
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\logistic_regression_pipeline.joblib

Training pipeline for: Random Forest
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\random_forest_pipeline.joblib

Training pipeline for: XGBoost
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\xgboost_pipeline.joblib

Training pipeline for: K-Nearest Neighbors
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\k_nearest_neighbors_pipeline.joblib

Training pipeline for: Gaussian Naive Bayes
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\gaussian_naive_bayes_pipeline.joblib

Training pipeline for: Bernoulli Naive Bayes
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\models\bernoulli_naive_bayes_pipeline.joblib

Training pipeline for: Decision Tree
✔ Saved pipeline to: C:\Users\KAUSHIK\Desktop\streamlit_app\model